# S7 — TEA: Temporal Experts Averaging (multi-seed)

One expert per year (2014, 2015, 2016), all from the same init (`set_seed(SEED)` before each). Averaged with recency weights (2014:0.5, 2015:0.75, 2016:1.0). Seeds 42, 1, 2.

In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))


In [2]:
import copy
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score
from huggingface_hub import snapshot_download
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset

DATA_DIR = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))

MODEL_NAME = 'roberta-base'
SEEDS      = [42, 1, 2]
STRATEGY   = 's7-tea'
RECENCY_W  = {'2014': 0.5, '2015': 0.75, '2016': 1.0}
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Device: cuda


## Data

In [3]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

with open(DATA_DIR / 'train_eval/train.json') as f:
    train_full = pd.DataFrame(json.load(f)).rename(columns={'distant_label': 'label'})
train_full['year'] = train_full['created_at'].str.extract(r'(20\d\d)')
print(train_full['year'].value_counts().sort_index())

eval_df = load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label')
test_dfs = {
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}

label2id = {l: i for i, l in enumerate(sorted(train_full['label'].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Labels:', label2id)


year
2014    16523
2015    15013
2016    18065
2017        7
Name: count, dtype: int64
Labels: {'negative': 0, 'positive': 1}


## Training loop (seeds 42, 1, 2)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    d = Dataset.from_dict({'text': df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

eval_ds  = make_dataset(eval_df)
test_ds  = {n: make_dataset(df) for n, df in test_dfs.items()}

all_results  = {}
saved_models = {}

for SEED in SEEDS:
    print(f'\n{"="*50}  SEED {SEED}')

    def train_expert(year):
        sub = train_full[train_full['year'] == year]
        ds_yr = make_dataset(sub)
        set_seed(SEED)   # identical init for every expert (required for weight averaging)
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)
        args = TrainingArguments(
            output_dir=f'/tmp/s7_seed{SEED}/{year}', num_train_epochs=25,
            per_device_train_batch_size=32, per_device_eval_batch_size=64,
            learning_rate=2e-05, warmup_ratio=0.1, weight_decay=0.01,
            eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
            logging_steps=50, fp16=(DEVICE=='cuda'), seed=SEED, report_to='none')
        tr = Trainer(model=m, args=args, train_dataset=ds_yr, eval_dataset=eval_ds,
                     processing_class=tokenizer,
                     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
        tr.train()
        return m

    experts = {yr: train_expert(yr) for yr in ['2014', '2015', '2016']}

    # Recency-weighted average
    total = sum(RECENCY_W.values())
    avg_sd = copy.deepcopy(experts['2016'].state_dict())
    for k in avg_sd:
        avg_sd[k] = sum(RECENCY_W[yr] * experts[yr].state_dict()[k].float()
                        for yr in RECENCY_W) / total
    avg_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)
    avg_model.load_state_dict(avg_sd)
    avg_model.to(DEVICE)

    eval_trainer = Trainer(model=avg_model,
                           args=TrainingArguments(output_dir='/tmp/s7_eval',
                                                   per_device_eval_batch_size=64,
                                                   fp16=(DEVICE=='cuda'), report_to='none'),
                           processing_class=tokenizer)

    seed_results = {}
    for name, ds in test_ds.items():
        preds_out = eval_trainer.predict(ds)
        f1 = f1_score(np.array(ds['label']), np.argmax(preds_out.predictions, axis=1), average='macro')
        seed_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = avg_model


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  test_within: F1=0.7473


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_short: F1=0.6938


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_long: F1=0.7048


## Summary

In [5]:
# ── Summary ────────────────────────────────────────────────
import numpy as np

splits = ['test_within', 'test_short', 'test_long']
print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short']  - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']   - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")

print()
for sp in splits:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals,ddof=1):.4f}  [{min(vals):.4f}–{max(vals):.4f}]")


    seed   within    short     long   RPD_short   RPD_long
      42   0.7253   0.6919   0.7046     -0.0461    -0.0285
       1   0.7356   0.6904   0.7135     -0.0615    -0.0300
       2   0.7473   0.6938   0.7048     -0.0716    -0.0569

test_within: mean=0.7361  std=0.0110  [0.7253–0.7473]
test_short: mean=0.6920  std=0.0017  [0.6904–0.6938]
test_long: mean=0.7077  std=0.0051  [0.7046–0.7135]


## Save to HuggingFace

In [6]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Saved: tamarasuarezrod/longeval-s7-tea-seed2
